# Cryptocurrency Volatility Analysis
### FIN 4600 · Fintech Foundations — Pandas & Risk Lab

**Goal.** Use a free public API and `pandas` to measure and compare the risk of five cryptocurrencies, then connect the numbers to real risk-management ideas: volatility, correlation and diversification, Value at Risk, Sharpe ratio, drawdown, and volatility clustering.

**What you will practice**

| Pandas skill | Where |
|---|---|
| Building a `DataFrame` from API JSON, datetime index | Tasks 1–2 |
| `.pct_change()`, `.dropna()`, alignment of several series | Tasks 2–3 |
| `.std()`, `.rolling()`, `.quantile()`, `.corr()`, `.cummax()`, `.autocorr()` | Tasks 4–10 |
| Sorting, ranking, assembling summary tables | Tasks 6, 8, 9 |
| Plotting with `matplotlib` / `seaborn` | Tasks 5–7, 10–11 |

**How this notebook works.** Cells marked **✏️ Task** have a `# YOUR CODE HERE` section. Replace the placeholder (`None` or `pass`) with your own code. The cell after each task runs automatic checks and prints ✅ or ❌. Passing the checks means your code is *plausible*, not that your interpretation is right; the discussion questions at the end carry a lot of weight.

**To submit:** *Kernel → Restart & Run All*, confirm there are no errors, save, and submit the `.ipynb` file with outputs visible.

## Part 0 · Setup

Install the libraries once from a terminal in your lab folder (we are using `uv` and/or VS Code, not Colab):

```bash
uv pip install -r requirements.txt
# or: pip install requests pandas numpy matplotlib seaborn yfinance
```

Then run the cell below.

In [ ]:
import os
import time
from pathlib import Path
from statistics import NormalDist

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import seaborn as sns

# ---- Lab configuration (do not change) ----
BASE_URL = "https://api.coingecko.com/api/v3"
DAYS = 365              # free public API: maximum 365 days of history
TRADING_DAYS = 365      # crypto trades every day of the year (stocks: ~252)
RISK_FREE = 0.02        # assumed annual risk-free rate (2%)
COINS = {               # display name -> CoinGecko id
    "Bitcoin": "bitcoin",
    "Ethereum": "ethereum",
    "Cardano": "cardano",
    "Solana": "solana",
    "Chainlink": "chainlink",
}
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Optional: a free CoinGecko "Demo" API key raises the rate limit.
API_KEY = os.getenv("COINGECKO_API_KEY")
HEADERS = {"x-cg-demo-api-key": API_KEY} if API_KEY else {}

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110


def check(label, test, hint=""):
    """Run test() (a function returning True/False) and print a pass/fail line."""
    try:
        ok = bool(test())
    except Exception as err:          # unfinished code lands here, not as a traceback
        ok, hint = False, f"{hint} ({type(err).__name__}: {err})".strip()
    print(("✅ PASS  " if ok else "❌ FAIL  ") + label)
    if not ok and hint:
        print("         hint:", hint)
    return ok

print("Setup complete. pandas", pd.__version__)

## Part 1 · Getting data from the CoinGecko API

CoinGecko offers free market data with no account required. We use one endpoint:

```
GET https://api.coingecko.com/api/v3/coins/{id}/market_chart?vs_currency=usd&days=365
```

It returns JSON with three lists, each made of `[timestamp_in_milliseconds, value]` pairs:

```python
{"prices":        [[1700006400000, 36500.1], [1700092800000, 37100.9], ...],
 "market_caps":   [[1700006400000, 7.1e11],  ...],
 "total_volumes": [[1700006400000, 1.9e10],  ...]}
```

Two details to be careful about: timestamps are **milliseconds** since 1970 (UTC), and the last observation is *today's partial day*, so the API can return two points that fall on the same calendar date.

### ✏️ Task 1 — Turn the JSON into a clean DataFrame

Complete `fetch_crypto_data`. It must return a `DataFrame` with
* a `DatetimeIndex` named `date`, **normalized to midnight** (one row per calendar day, no duplicates; if a date appears twice keep the *last* one),
* columns `price`, `market_cap`, `volume`.

In [ ]:
def fetch_crypto_data(coin_id, days=DAYS):
    """Download daily USD price, market cap and volume for one coin."""
    url = f"{BASE_URL}/coins/{coin_id}/market_chart"
    params = {"vs_currency": "usd", "days": days}

    # YOUR CODE HERE
    # 1. response = requests.get(url, params=params, headers=HEADERS, timeout=30)
    # 2. response.raise_for_status()   # raises an error on HTTP 4xx/5xx (e.g. 429 = rate limit)
    # 3. data = response.json()
    # 4. build a DataFrame from data["prices"], data["market_caps"], data["total_volumes"]
    #    (each item is [timestamp_ms, value]); use pd.to_datetime(..., unit="ms")
    # 5. normalize the index to midnight and drop duplicate dates (keep the last)
    pass

**Offline test of Task 1.** This feeds your function a tiny fake API response (four points, the last two on the same day) so you can verify your parsing without using the network.

In [ ]:
_payload = {
    "prices":        [[1700006400000, 100.0], [1700092800000, 110.0], [1700179200000, 121.0], [1700200000000, 125.0]],
    "market_caps":   [[1700006400000, 1e9],   [1700092800000, 1.1e9], [1700179200000, 1.2e9], [1700200000000, 1.3e9]],
    "total_volumes": [[1700006400000, 5e6],   [1700092800000, 6e6],   [1700179200000, 7e6],   [1700200000000, 8e6]],
}

class _FakeResponse:
    def raise_for_status(self): pass
    def json(self): return _payload

_real_get = requests.get
requests.get = lambda *a, **k: _FakeResponse()
try:
    test_df = fetch_crypto_data("bitcoin", 3)
except Exception as err:
    test_df = None
    print("fetch_crypto_data raised:", type(err).__name__, err)
finally:
    requests.get = _real_get

check("returns a DataFrame", lambda: isinstance(test_df, pd.DataFrame), "return df at the end of the function")
check("columns are price, market_cap, volume", lambda: list(test_df.columns) == ["price", "market_cap", "volume"])
check("index is a DatetimeIndex named 'date'", lambda: isinstance(test_df.index, pd.DatetimeIndex) and test_df.index.name == "date")
check("duplicate dates removed (3 rows, not 4)", lambda: len(test_df) == 3, "use index.duplicated(keep='last')")
check("keeps the LAST value for a repeated date", lambda: test_df["price"].iloc[-1] == 125.0)
check("index normalized to midnight", lambda: (test_df.index == test_df.index.normalize()).all(), "use .normalize()")
check("market cap and volume are the right series", lambda: test_df["market_cap"].iloc[0] == 1e9 and test_df["volume"].iloc[0] == 5e6,
      "market_caps -> market_cap, total_volumes -> volume (use item[1], the value, not item[0])")

### Download the data (provided)

`load_or_fetch` calls your function once per coin and **caches** the result as a CSV in `data/`, so repeated runs do not hit the API again (the free tier is rate-limited, and HTTP **429** means "slow down"). If your instructor provided a `data/` folder, the files load instantly. To force a fresh download use `load_or_fetch(coin, refresh=True)`.

In [ ]:
def load_or_fetch(coin_id, days=DAYS, refresh=False, retries=3):
    path = DATA_DIR / f"{coin_id}_{days}d.csv"
    if path.exists() and not refresh:
        return pd.read_csv(path, index_col="date", parse_dates=True)
    for attempt in range(1, retries + 1):
        try:
            df = fetch_crypto_data(coin_id, days)
            break
        except requests.HTTPError as err:
            status = err.response.status_code if err.response is not None else None
            if status == 429 and attempt < retries:
                wait = 30 * attempt
                print(f"  rate limited on {coin_id}; waiting {wait}s (attempt {attempt}/{retries})")
                time.sleep(wait)
            else:
                raise
    df.to_csv(path)
    time.sleep(2.5)          # be polite to the free API
    return df

btc = load_or_fetch("bitcoin")
print(f"Bitcoin: {len(btc)} rows, {btc.index.min().date()} to {btc.index.max().date()}")
btc.head()

## Part 2 · Prices and returns

### ✏️ Task 2 — Build one table of closing prices

Load all five coins (use `load_or_fetch`) and create a DataFrame `prices` with **one column per coin (named "Bitcoin", "Ethereum", …) and the date as the index**, containing only the `price` series. Then drop any rows that have missing values in *any* coin so every column covers exactly the same dates.

In [ ]:
raw = {}   # name -> DataFrame from load_or_fetch

# YOUR CODE HERE
prices = None

In [ ]:
check("prices has five columns in the order of COINS", lambda: list(prices.columns) == list(COINS))
check("index is a DatetimeIndex", lambda: isinstance(prices.index, pd.DatetimeIndex))
check("no missing values", lambda: prices.isna().sum().sum() == 0, "try .dropna()")
check("at least 300 daily observations", lambda: len(prices) >= 300)
check("index is sorted and unique", lambda: prices.index.is_monotonic_increasing and prices.index.is_unique)
prices.tail()

### ✏️ Task 3 — Daily returns

Create two DataFrames with the same columns as `prices`:

* `returns` — **simple** daily returns, $r_t = \dfrac{P_t}{P_{t-1}} - 1$
* `log_returns` — **log** returns, $\ln\dfrac{P_t}{P_{t-1}}$

Drop the first (empty) row of each. Everything later in the lab uses `returns`.

In [ ]:
# YOUR CODE HERE
returns = None
log_returns = None

In [ ]:
check("returns has one fewer row than prices", lambda: len(returns) == len(prices) - 1)
check("no NaN values in returns", lambda: returns.isna().sum().sum() == 0)
check("first Bitcoin return is correct",
      lambda: np.isclose(returns["Bitcoin"].iloc[0], prices["Bitcoin"].iloc[1] / prices["Bitcoin"].iloc[0] - 1))
check("log returns are close to simple returns for small moves",
      lambda: np.isclose(log_returns["Bitcoin"].iloc[0], np.log(1 + returns["Bitcoin"].iloc[0])))
returns.describe().T

**Look at the table above.** Compare each coin's `std`, `min` and `max`. You will quantify this properly in the next part.

## Part 3 · Measuring volatility

**Volatility** is the standard deviation of returns. Daily volatility is converted to an annual figure by multiplying by $\sqrt{N}$, where $N$ is the number of return observations per year. Crypto trades 365 days a year, so here $N = 365$ (for stocks you would use about 252).

### ✏️ Task 4 — Volatility metrics for one asset

Complete `volatility_metrics(r)`, which takes a Series of daily returns and returns a dictionary with these keys:

| key | meaning |
|---|---|
| `daily_vol` | standard deviation of daily returns |
| `annual_vol` | `daily_vol * sqrt(periods)` |
| `worst_day` | the minimum daily return |
| `best_day` | the maximum daily return |
| `range` | `best_day - worst_day` |

Then build `vol_table`, a DataFrame with one **row per coin** and those five columns.

In [ ]:
def volatility_metrics(r, periods=TRADING_DAYS):
    r = r.dropna()
    # YOUR CODE HERE
    return {}

vol_table = None   # YOUR CODE HERE: rows = coins, columns = the five metrics

In [ ]:
check("vol_table has 5 rows and the 5 required columns",
      lambda: vol_table.shape == (5, 5) and set(vol_table.columns) == {"daily_vol", "annual_vol", "worst_day", "best_day", "range"})
check("annual_vol = daily_vol * sqrt(365)", lambda: np.allclose(vol_table["annual_vol"], vol_table["daily_vol"] * np.sqrt(365)),
      "annualize with np.sqrt(periods)")
check("daily_vol matches pandas .std()", lambda: np.isclose(vol_table.loc["Bitcoin", "daily_vol"], returns["Bitcoin"].std()))
check("worst_day is negative and best_day is positive", lambda: (vol_table["worst_day"] < 0).all() and (vol_table["best_day"] > 0).all())
vol_table

### ✏️ Task 5 — Rolling volatility

A single number hides how risk changes over time. Compute `rolling_vol`: the **30-day rolling standard deviation of daily returns, annualized** (same shape as `returns`; the first 29 rows will be `NaN`). Then plot all five coins on one chart.

In [ ]:
# YOUR CODE HERE
rolling_vol = None

In [ ]:
check("same columns as returns", lambda: list(rolling_vol.columns) == list(returns.columns))
check("first 29 rows are NaN, 30th is a number",
      lambda: rolling_vol.iloc[:29].isna().all().all() and rolling_vol.iloc[29].notna().all(), "use .rolling(window=30)")
check("values are annualized (typically between 0.1 and 3.0)", lambda: 0.1 < rolling_vol.stack().median() < 3.0, "multiply by np.sqrt(365)")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
# YOUR CODE HERE: plot rolling_vol on ax, then add a title and axis labels
plt.show()

### ✏️ Task 6 — Rank the coins by risk

Create `ranking`: `vol_table` sorted from **least to most volatile** by `annual_vol`, with an extra column `x_bitcoin` showing each coin's annual volatility as a multiple of Bitcoin's. Then draw a horizontal bar chart of annual volatility.

In [ ]:
# YOUR CODE HERE
ranking = None

In [ ]:
check("sorted ascending by annual_vol", lambda: ranking["annual_vol"].is_monotonic_increasing, "sort_values('annual_vol')")
check("x_bitcoin is 1.0 for Bitcoin", lambda: np.isclose(ranking.loc["Bitcoin", "x_bitcoin"], 1.0))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
# YOUR CODE HERE: horizontal bar chart of ranking["annual_vol"], with title and labelled axis
plt.show()
ranking

## Part 4 · Correlation and diversification

If assets do not move together, combining them lowers risk. The **correlation matrix** of returns tells you how much they move together (+1 = perfectly together, 0 = unrelated, −1 = opposite).

### ✏️ Task 7 — Correlation and an equal-weight portfolio

1. `corr` — the correlation matrix of `returns`, and a heatmap of it.
2. `avg_pairwise_corr` — the average of the **off-diagonal** correlations (exclude the 1.0s on the diagonal).
3. `portfolio_returns` — daily returns of an **equal-weight portfolio** (20% in each coin, rebalanced daily): the row-wise mean of `returns`.
4. `portfolio_annual_vol` — its annualized volatility, and `avg_individual_vol` — the average of the five individual annualized volatilities.

In [ ]:
# YOUR CODE HERE
corr = None
avg_pairwise_corr = None
portfolio_returns = None
portfolio_annual_vol = None
avg_individual_vol = None

In [ ]:
check("corr is a symmetric 5x5 matrix with ones on the diagonal",
      lambda: corr.shape == (5, 5) and np.allclose(np.diag(corr), 1) and np.allclose(corr, corr.T))
check("avg_pairwise_corr excludes the diagonal (so it is below 1)", lambda: -1 < avg_pairwise_corr < 0.999)
check("portfolio_returns is a Series with one value per day", lambda: isinstance(portfolio_returns, pd.Series) and len(portfolio_returns) == len(returns),
      "use returns.mean(axis=1)")
check("portfolio volatility is below the average individual volatility", lambda: portfolio_annual_vol < avg_individual_vol)
print(f"Average pairwise correlation : {avg_pairwise_corr:.2f}")
print(f"Equal-weight portfolio vol    : {portfolio_annual_vol:.1%}")
print(f"Average individual vol        : {avg_individual_vol:.1%}")
print(f"Diversification benefit       : {1 - portfolio_annual_vol / avg_individual_vol:.1%} lower than the average coin")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
# YOUR CODE HERE: seaborn heatmap of corr (annotate with 2 decimals, range -1 to 1)
plt.show()

## Part 5 · Tail risk and risk-adjusted return

### ✏️ Task 8 — Value at Risk and Expected Shortfall

**Value at Risk (VaR)** answers: *"On a bad day, how much could I lose?"* The 95% one-day historical VaR is the loss that was exceeded on only 5% of days. **Expected Shortfall (ES)** is the *average* loss on those worst 5% of days.

We report both as **positive numbers** (a VaR of `0.05` means a 5% loss).

Complete the three functions, where `level` is the confidence level:

* `historical_var(r, level)` — minus the `(1 - level)` quantile of `r`
* `expected_shortfall(r, level)` — minus the mean of the returns at or below that quantile
* `normal_var(r, level)` — VaR *assuming returns are normally distributed*: $-(\mu + z\sigma)$ where $z$ = `NormalDist().inv_cdf(1 - level)`

Then build `var_table` (one row per coin) with columns `VaR 95%`, `VaR 99%`, `ES 95%`, `Normal VaR 99%`.

In [ ]:
def historical_var(r, level=0.95):
    # YOUR CODE HERE
    pass

def expected_shortfall(r, level=0.95):
    # YOUR CODE HERE
    pass

def normal_var(r, level=0.95):
    # YOUR CODE HERE
    pass

var_table = None   # YOUR CODE HERE

In [ ]:
r_btc = returns["Bitcoin"]
check("historical VaR is a positive loss", lambda: historical_var(r_btc) > 0, "remember the minus sign")
check("about 5% of days are worse than -VaR (95%)", lambda: abs((r_btc < -historical_var(r_btc)).mean() - 0.05) < 0.01)
check("99% VaR is larger than 95% VaR", lambda: historical_var(r_btc, 0.99) > historical_var(r_btc, 0.95))
check("Expected Shortfall >= VaR at the same level", lambda: expected_shortfall(r_btc) >= historical_var(r_btc))
check("normal_var returns a positive number", lambda: normal_var(r_btc, 0.99) > 0)
check("var_table is 5 x 4 with the right columns",
      lambda: var_table.shape == (5, 4) and list(var_table.columns) == ["VaR 95%", "VaR 99%", "ES 95%", "Normal VaR 99%"])
var_table.style.format("{:.2%}")

**Dollar view.** A $10,000 position in each coin would have lost at least this much on the worst 5% of days:

In [ ]:
(var_table["VaR 95%"] * 10_000).round(0).rename("Loss on a $10,000 position (95% VaR)").to_frame().style.format("${:,.0f}")

### ✏️ Task 9 — Sharpe ratio and maximum drawdown

**Sharpe ratio** = excess return per unit of risk:

$$\text{Sharpe} = \frac{\bar r \cdot N - r_f}{\sigma \cdot \sqrt{N}}, \qquad N = 365,\; r_f = 2\%$$

where $\bar r$ and $\sigma$ are the mean and standard deviation of *daily* returns.

**Drawdown** is the percentage fall from the highest price reached so far: $\dfrac{P_t}{\max_{s\le t} P_s} - 1$. The **maximum drawdown** is the worst of these.

Complete `sharpe_ratio(r, rf, periods)`. Then compute `drawdown` (a DataFrame shaped like `prices`) and `max_dd` (a Series: the minimum drawdown of each coin).

In [ ]:
def sharpe_ratio(r, rf=RISK_FREE, periods=TRADING_DAYS):
    r = r.dropna()
    # YOUR CODE HERE
    pass

# YOUR CODE HERE
drawdown = None
max_dd = None

In [ ]:
r_btc = returns["Bitcoin"]
check("with rf = 0, Sharpe = mean/std * sqrt(365)",
      lambda: np.isclose(sharpe_ratio(r_btc, rf=0), r_btc.mean() / r_btc.std() * np.sqrt(365)))
check("a higher risk-free rate lowers the Sharpe ratio", lambda: sharpe_ratio(r_btc, rf=0.05) < sharpe_ratio(r_btc, rf=0.0))
check("drawdown has the same shape as prices", lambda: drawdown.shape == prices.shape)
check("drawdown is never positive and equals 0 at the first date", lambda: (drawdown <= 1e-12).all().all() and np.allclose(drawdown.iloc[0], 0),
      "use prices / prices.cummax() - 1")
check("max_dd has one negative value per coin", lambda: len(max_dd) == 5 and (max_dd < 0).all())

fig, ax = plt.subplots(figsize=(11, 4))
drawdown.plot(ax=ax, linewidth=1.2)
ax.set_title("Drawdown from running peak")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend(title="", ncol=5, loc="lower left")
plt.show()

### Risk-return summary (provided)

This cell pulls your earlier results into one table. If it errors, an earlier task is unfinished.

In [ ]:
summary = pd.DataFrame({
    "Annual return": returns.mean() * TRADING_DAYS,
    "Annual vol": vol_table["annual_vol"],
    "Sharpe": pd.Series({c: sharpe_ratio(returns[c]) for c in returns}),
    "VaR 95% (1d)": var_table["VaR 95%"],
    "ES 95% (1d)": var_table["ES 95%"],
    "Max drawdown": max_dd,
}).loc[list(COINS)]
summary.style.format({"Annual return": "{:.1%}", "Annual vol": "{:.1%}", "Sharpe": "{:.2f}",
                      "VaR 95% (1d)": "{:.2%}", "ES 95% (1d)": "{:.2%}", "Max drawdown": "{:.1%}"})

## Part 6 · Volatility clustering and the risk-return picture

### ✏️ Task 10 — Does volatility cluster?

Financial returns are almost impossible to predict from yesterday's return, but their **size** is predictable: big moves tend to follow big moves. You can see this with **autocorrelation** (the correlation of a series with a lagged copy of itself).

For Bitcoin (`COIN = "Bitcoin"`), compute two lists for lags 1 to 10 using `Series.autocorr(lag=k)`:

* `acf_returns` — autocorrelation of the returns
* `acf_abs` — autocorrelation of the **absolute** returns (a simple proxy for volatility)

Then draw them as two bar charts side by side.

In [ ]:
COIN = "Bitcoin"
lags = list(range(1, 11))

# YOUR CODE HERE
acf_returns = None
acf_abs = None

In [ ]:
check("two lists of 10 numbers", lambda: len(acf_returns) == 10 and len(acf_abs) == 10)
check("autocorrelations are between -1 and 1", lambda: all(-1 <= v <= 1 for v in acf_returns + acf_abs))
check("lag-1 autocorrelation of absolute returns is higher than of raw returns", lambda: acf_abs[0] > acf_returns[0],
      "if this fails on a quiet sample, still read the plots carefully")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
# YOUR CODE HERE: bar chart of acf_returns on axes[0] and acf_abs on axes[1]
# Give each panel a title and label the x-axis "lag (days)".
plt.show()

### ✏️ Task 11 — Risk-return scatter plot

Using `summary`, make a scatter plot with **annual volatility on the x-axis**, **annual return on the y-axis**, point **color = Sharpe ratio**, a label with each coin's name next to its point, a colorbar, a horizontal line at 0, and titles/labels.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
# YOUR CODE HERE
plt.show()

## Part 7 · Comparison with a traditional asset (optional but recommended)

This cell downloads the S&P 500 ETF (**SPY**) for the same dates with `yfinance`, as in your first lab. If you are offline it is skipped automatically.

### ✏️ Task 12 — Annualize correctly

Compute `spy_annual_vol` from `spy_returns`. Stock markets are open about **252** days a year, so use $\sqrt{252}$ rather than $\sqrt{365}$.

In [ ]:
try:
    import yfinance as yf
    _px = yf.download("SPY", start=prices.index[0], end=prices.index[-1] + pd.Timedelta(days=1),
                      auto_adjust=True, progress=False)["Close"].squeeze()
    spy_returns = _px.pct_change().dropna()
    SPY_OK = len(spy_returns) > 100
except Exception as err:
    SPY_OK = False
    print("SPY data unavailable, skipping Task 12:", type(err).__name__)
print("SPY data loaded:" , SPY_OK)

In [ ]:
spy_annual_vol = None   # YOUR CODE HERE (only if SPY_OK)

In [ ]:
if SPY_OK:
    check("SPY annual vol is a plausible number (3% to 60%)", lambda: 0.03 < spy_annual_vol < 0.60, "use np.sqrt(252)")
    comparison = pd.concat([summary["Annual vol"], pd.Series({"S&P 500 (SPY)": spy_annual_vol})]).sort_values()
    print((comparison.apply(lambda v: f"{v:.1%}")).to_string())
    print(f"\nBitcoin is {summary.loc['Bitcoin', 'Annual vol'] / spy_annual_vol:.1f}x as volatile as the S&P 500 over this period.")
else:
    print("Skipped (no SPY data).")

## Part 8 · Discussion questions

Answer each question in **2–5 sentences, citing specific numbers from your tables or charts** (for example "Solana's annual volatility was X times Bitcoin's"). Double-click the answer cell, replace the placeholder text, and run it. Answers with no numbers will receive little credit.

**Q1. Volatility ranking.** Which coin was the most volatile and which was the least? Give the annualized volatilities. What features of these assets (size, age, use case, liquidity) might explain the ranking?

*Your answer here.*

**Q2. Diversification.** Report the average pairwise correlation and compare the equal-weight portfolio's volatility with the average individual volatility. Does holding all five coins provide strong diversification? Why or why not?

*Your answer here.*

**Q3. Normal vs. real tails.** Compare `VaR 99%` with `Normal VaR 99%`, and `ES 95%` with `VaR 95%`. What does this tell you about the shape of crypto return distributions? Why can relying on a normal-distribution VaR be dangerous?

*Your answer here.*

**Q4. Risk-adjusted performance.** Which coin had the best Sharpe ratio and which the worst? Is the coin with the highest return also the one with the best Sharpe ratio? What would change if the risk-free rate were 5%?

*Your answer here.*

**Q5. Volatility clustering.** Describe the two autocorrelation charts. Why is it significant that the autocorrelation of absolute returns is much larger than that of raw returns? How could a risk manager use this fact?

*Your answer here.*

**Q6. Drawdown and the investor's experience.** Look at the maximum drawdown column. Why might an investor care more about drawdown than about annualized volatility? Which coin would have been hardest to hold through, and why?

*Your answer here.*

**Q7. Traditional assets and honest limitations.** (a) If you ran Task 12, how does Bitcoin's volatility compare to SPY's, and why must you use 252 for SPY but 365 for crypto? (b) List **two limitations** of this analysis (think about sample length, the period chosen, survivorship, data source, and the assumptions behind VaR and Sharpe).

*Your answer here.*

## Optional extension challenges

*(Not required; mention them in your answers if you try one.)*

1. **EWMA volatility.** Replace the rolling standard deviation with `returns.ewm(span=30).std()` and compare the two lines. Which reacts faster to a shock?
2. **Rolling correlation.** Plot the 60-day rolling correlation between Bitcoin and each other coin. Does it rise in turbulent periods?
3. **Weekly vs daily.** Resample prices to weekly (`prices.resample("W").last()`), recompute annualized volatility with $\sqrt{52}$, and compare. Does volatility scale with the square root of time as theory predicts?
4. **Minimum-variance weights.** Find portfolio weights (summing to 1, no shorting) that minimize portfolio volatility using the covariance matrix `returns.cov()`.

## Submission checklist and grading

- [ ] *Kernel → Restart & Run All* finishes with no errors; outputs are visible.
- [ ] Every ✅ check passes (or you explain in a comment why not).
- [ ] All plots have a title and labelled axes.
- [ ] All seven discussion questions are answered with specific numbers.

| Component | Points |
|---|---|
| Code correctness (Tasks 1–12, automatic checks) | 50 |
| Visualizations (Tasks 5, 6, 7, 9, 10, 11): clear, labelled | 15 |
| Discussion answers (Q1–Q7, 25 points total) | 25 |
| Reproducibility and clean notebook (runs top to bottom, readable code) | 10 |

*Data: CoinGecko API (free tier). Prices are aggregated across exchanges and are for education only; nothing here is investment advice.*